# Local library call demo (`piboufilings`)

This notebook runs against the local source tree (no package install required).

Default storage backend is DuckDB; install it once with `pip install duckdb`
or pass `export_format="csv"` to `get_filings(...)` for the CSV fallback.


In [ ]:
import warnings 

#warnings.filterwarnings('ignore')

In [ ]:
from pathlib import Path
import sys

# Ensure local repo root is importable
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'Using repo root: {repo_root}')

In [ ]:
from piboufilings import get_parser_for_form_type_internal

# Local call on the library (no SEC network request)
parser_13f = get_parser_for_form_type_internal('13F-HR', './demo_local_output')
parser_nport = get_parser_for_form_type_internal('NPORT-P', './demo_local_output')
parser_sec16 = get_parser_for_form_type_internal('SECTION-6', './demo_local_output')

print('13F parser:', type(parser_13f).__name__)
print('NPORT parser:', type(parser_nport).__name__)
print('Section16 parser:', type(parser_sec16).__name__)

## Apple insider trading example

This downloads two complementary datasets and then combines them below:

- **SECTION-6** (Forms 3/4/5) for **Apple Inc.** (CIK `0000320193`) — every insider transaction reported by Apple officers and directors.
- **13F-HR** for **Berkshire Hathaway** (CIK `0001067983`) — Berkshire's institutional holdings, used in the second query to inspect their largest positions (Apple has historically been one of them).

Set `RUN_NETWORK = True` and provide a real contact email before executing.

In [ ]:
RUN_NETWORK = True

# Apple Inc. (issuer of insider trades) and Berkshire Hathaway (13F filer).
apple_cik = "0000320193"
berkshire_cik = "0001067983"
insider_cik = [apple_cik, berkshire_cik]

if RUN_NETWORK:
    from piboufilings import get_filings

    get_filings(
        user_name="Your Name",
        user_agent_email="your@email.com",
        cik=insider_cik,
        form_type=["SECTION-6", "13F-HR"],
        start_year=2025,
        end_year=2025,
        base_dir="./demo_local_output",
        log_dir="./demo_local_logs",
        raw_data_dir="./demo_local_raw",
        max_workers=10,
        keep_raw_files=False,
        export_format="duckdb",   # or "csv" if you do not want the duckdb dependency
    )
    print("Download/parse completed.")
else:
    print("Network call skipped. Set RUN_NETWORK=True to execute.")


# Exploring the db

In [ ]:
# Peek at the DuckDB output written above.
#
# Note: if you re-run the previous cell (`get_filings`), make sure this cell's
# `con` from a prior run is not still open — re-running this cell is safe;
# restarting the kernel is the simplest reset.
import duckdb
from pathlib import Path

db_path = Path("./demo_local_output/piboufilings.duckdb")
if db_path.exists():
    con = duckdb.connect(str(db_path))
    try:
        tables = [row[0] for row in con.execute(
            "SELECT table_name FROM information_schema.tables ORDER BY table_name"
        ).fetchall()]
        print("Tables:", tables)
        for t in tables:
            n = con.execute(f'SELECT COUNT(*) FROM "{t}"').fetchone()[0]
            print(f"  {t}: {n:,} rows")
    finally:
        print("DuckDB connection opened.")
else:
    print(f"No DuckDB file at {db_path}; rerun the cell above with RUN_NETWORK=True.")


In [ ]:
# Apple insider trading: most recent reported transactions from SECTION-6.
query = f"""
SELECT
    t.*,
    f.PERIOD_OF_REPORT,
    f.ISSUER_NAME
FROM transactions_sec16 t
JOIN filing_info_sec16 f ON t.ACCESSION_NUMBER = f.ACCESSION_NUMBER
WHERE f.ISSUER_CIK = '{apple_cik}'
"""

transactions_df = con.execute(query).df()

transactions_df[[
    "DOCUMENT_TYPE",
    "PERIOD_OF_REPORT",
    "ISSUER_TRADING_SYMBOL",
    "RPT_OWNER_NAME",
    "TRANSACTION_CODE",
    "TRANSACTION_SHARES",
    "TRANSACTION_PRICE_PER_SHARE",
    "SHARES_OWNED_FOLLOWING_TRANSACTION",
]].sort_values("PERIOD_OF_REPORT", ascending=False).head()

In [ ]:
# Berkshire Hathaway's most recent 13F-HR holdings (top positions by share value).
query = f"""
SELECT * FROM holdings_13f
WHERE SEC_FILE_NUMBER = (
    SELECT SEC_FILE_NUMBER
    FROM filing_info_13f
    WHERE CIK = '{berkshire_cik}'
    ORDER BY CONFORMED_DATE DESC
    LIMIT 1
)
"""

holdings_df = con.execute(query).df()

max_date = holdings_df["CONFORMED_DATE"].max()
holdings_df = holdings_df[holdings_df["CONFORMED_DATE"] == max_date]
holdings_df.sort_values("SHARE_VALUE", ascending=False).head(10)